# LangGraph planner agent + gemma3:1b

Ноутбук рассчитан на запуск внутри Docker Compose. Он использует Ollama по адресу `http://ollama:11434` и модель `gemma3:1b`.


In [1]:
import os
from typing import List
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END


In [4]:
import requests

resp = requests.post(
    "http://ollama:11434/api/pull",
    json={"model": "gemma3:1b", "stream": False},
    timeout=3600
)

print(resp.status_code)
print(resp.text[:2000])

200
{"status":"success"}


In [5]:
import json

resp = requests.get("http://ollama:11434/api/tags", timeout=30)
print(resp.status_code)
print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:4000])

200
{
  "models": [
    {
      "name": "gemma3:1b",
      "model": "gemma3:1b",
      "modified_at": "2026-04-06T19:32:47.926694438Z",
      "size": 815319791,
      "digest": "8648f39daa8fbf5b18c7b4e6a8fb4990c692751d49917417b8842ca5758e7ffc",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "gemma3",
        "families": [
          "gemma3"
        ],
        "parameter_size": "999.89M",
        "quantization_level": "Q4_K_M"
      }
    }
  ]
}


In [6]:
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://ollama:11434')
MODEL_NAME = 'gemma3:1b'

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0
)

llm.invoke('Ответь одним словом: готов?')


AIMessage(content='Готов.\n', additional_kwargs={}, response_metadata={'model': 'gemma3:1b', 'created_at': '2026-04-06T19:34:00.560285262Z', 'done': True, 'done_reason': 'stop', 'total_duration': 29550309358, 'load_duration': 27853789411, 'prompt_eval_count': 18, 'prompt_eval_duration': 537966758, 'eval_count': 5, 'eval_duration': 302681389, 'logprobs': None, 'model_name': 'gemma3:1b', 'model_provider': 'ollama'}, id='lc_run--019d6449-2ef1-7491-bed0-4f3716ecacf7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 5, 'total_tokens': 23})

In [7]:
class TaskPlan(BaseModel):
    goal: str = Field(description='Краткая цель')
    subtasks: List[str] = Field(description='Список последовательных подзадач')
    first_step: str = Field(description='Первый шаг на 5 минут')
    checklist: List[str] = Field(description='Короткий чек-лист')

structured_llm = llm.with_structured_output(TaskPlan)


In [8]:
SYSTEM_PROMPT = '''
Ты — ассистент по планированию.
Разбивай задачу на 3-7 подзадач.
Каждая подзадача должна быть короткой и начинаться с глагола.
Не добавляй лишних объяснений.
В конце всегда выделяй первый шаг на 5 минут и чек-лист.
'''

def planner_node(state: MessagesState):
    user_text = state['messages'][-1].content
    result = structured_llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_text),
    ])
    return {'messages': [HumanMessage(content=result.model_dump_json(indent=2, ensure_ascii=False))]}

graph = StateGraph(MessagesState)
graph.add_node('planner', planner_node)
graph.add_edge(START, 'planner')
graph.add_edge('planner', END)
app = graph.compile()


In [9]:
task = 'Подготовить план дня: диплом, ответы на письма, спорт и бытовые дела'
result = app.invoke({'messages': [HumanMessage(content=task)]})
print(result['messages'][-1].content)


{
  "goal": "Составить план дня для учебы, ответов на письма, спорта и бытовых дел",
  "subtasks": [
    "Определить приоритеты задач",
    "Запланировать время для каждого занятия",
    "Запланировать время для отдыха и восстановления",
    "Запланировать время для спорта",
    "Запланировать время для бытовых дел",
    "Проверить и отредактировать план",
    "], 5,  ",
    "Checklist:",
    "1. Определить приоритеты задач (5 минут)",
    "2. Запланировать время для каждого занятия (10 минут)",
    "3. Запланировать время для отдыха и восстановления (5 минут)",
    "4. Запланировать время для спорта (10 минут)",
    "5. Запланировать время для бытовых дел (5 минут)",
    "6. Проверить и отредактировать план (5 минут)",
    "7.  Зафиксировать план дня (10 минут)",
    "7.  Завершить"
  ],
  "first_step": "10 минут",
  "checklist": [
    "Определить приоритеты задач (10 минут)"
  ]
}
